In [ ]:
%%capture
%pip install ollama beautifulsoup4 requests youtube-transcript-api

In [7]:
import requests
from bs4 import BeautifulSoup
import json
from youtube_transcript_api import YouTubeTranscriptApi, RequestBlocked
from youtube_transcript_api.formatters import JSONFormatter
from time import sleep
import pathlib
import ollama

In [5]:
PATH_RAW = "../data/rag_text/raw_yt_transcripts.json"
PATH_CLEANED = "../data/rag_text/cleaned_yt_transcripts.json"

In [11]:
def clean_text(text):
  text = text.replace("\n", " ")
  return text

*Prerequisite:* This notebook requires a local instance of ollama running.  
Make sure to activate Ollama and have the model ``llama3.1:8b`` installed.

# Scraping of Forza Horizon Tuning Forums
*Das HF_TOKEN muss mit `setx HF_TOKEN "TOKENWERT"` gesetzt werden*

In [13]:
def fetch_threads_list(category_url):
    resp = requests.get(category_url + ".json")
    data = resp.json()
    threads = []
    for t in data["topic_list"]["topics"]:
        threads.append({
            "topic_id": t["id"],
            "topic_slug": t["slug"],
            "title": t["title"]
        })
    return threads

def fetch_thread_posts(topic_id, topic_slug):
    url = f"https://forums.forza.net/t/{topic_slug}/{topic_id}.json"
    resp = requests.get(url)
    data = resp.json()
    posts = []
    for p in data["post_stream"]["posts"]:
        posts.append({
            "post_id": p["id"],
            "author_name": p["name"],
            "username": p["username"],
            "created_at": p["created_at"],
            "content_text": clean_text(BeautifulSoup(p["cooked"], "html.parser").get_text()),
            "topic_id": p["topic_id"],
            "topic_slug": p["topic_slug"]
        })
    return posts

def scrape_category(category_url):
    threads = fetch_threads_list(category_url)
    forum_data = []

    for t in threads:
        posts = fetch_thread_posts(t["topic_id"], t["topic_slug"])
        forum_data.append({
            "topic_id": t["topic_id"],
            "topic_slug": t["topic_slug"],
            "title": t["title"],
            "posts": posts
        })

    return forum_data

In [16]:
CATEGORY_URL = "https://forums.forza.net/tags/c/community-hub/ugc/tuning/148/fh5"
forum_data = scrape_category(CATEGORY_URL)

with open("../data/rag_text/raw_threads.json", "w", encoding="utf-8") as f:
    json.dump(forum_data, f, ensure_ascii=False, indent=4)

# YT Videos about getting faster in FH5

In [ ]:
ytt_api = YouTubeTranscriptApi()
formatter = JSONFormatter()

system_prompt = """
    You are a professional editor. Your task is to clean YouTube transcripts. "
    "Rules: \n"
    "1. Add punctuation and capitalization.\n"
    "2. Correct spelling of car names and racing terms.\n"
    "3. DO NOT include any introductory or concluding text such as "Here is the cleaned transcript".\n"
    "4. Output ONLY the cleaned transcript.
    "5. Do not change the original meaning of the text.
    """

user_prompt = ""

In [ ]:
output_raw_transcripts = []
"""
Youtube Searches:
How to get faster in Forza Horizon 5.
How to win more races in Forza Horizon 5.
"""

video_ids = [
    "aq96SH5zy3Y", # Forza Horizon 5 Ultimate Beginner's Guide | Tips You Should Know
    "rAin4ZrmvhU", # Mastering Car Control in Forza Horizon 5 - Tips & Tricks!
    "V5PNNsE_KtQ", # Wanna Become Faster In Forza Horizon 5 ?? 10 Tips For YOU!
    "RIEgZ3xlliI", # How To Win Races ✅ How To Get Faster - Forza Horizon 5 Driving School - Racing Lines
    ]

for id in video_ids:
  print(f"Attempting to fetch raw transcript for Video ID: {id}")
  sleep(5) # to avoid rate limiting
  try:
    transcript = ytt_api.fetch(id)
    json_formatted = formatter.format_transcript(transcript)
    data = json.loads(json_formatted)

    texts = [entry["text"] for entry in data]
    full_text = " ".join(texts)

    output_raw_transcripts.append({"video_id": id, "full_text": full_text})
    print(f"Successfully fetched and processed raw transcript for {id}.")

  except RequestBlocked:
    print(f"RequestBlocked: Could not retrieve raw transcript for video ID {id}.")
    output_raw_transcripts.append({"video_id": id, "full_text": "RequestBlocked."})
  except Exception as e:
    print(f"An unexpected error occurred for video ID {id}: {e}.")
    output_raw_transcripts.append({"video_id": id, "full_text": "unexpected error."})


# Ensure the directory exists before saving
pathlib.Path("transcripts").mkdir(parents=True, exist_ok=True)

with open(PATH_RAW, 'w', encoding='utf-8') as f:
    json.dump(output_raw_transcripts, f, ensure_ascii=False, indent=2)

print(f"Raw transcripts (or placeholders) saved to {PATH_RAW}")

In [ ]:
with open(PATH_RAW, 'r', encoding='utf-8') as f:
    raw_yt_transcripts = json.load(f)

cleaned_transcripts = []

print("Starting cleaning process...")
for entry in raw_yt_transcripts:
  video_id = entry["video_id"]
  full_text = entry["full_text"]

  if full_text == "RequestBlocked.":
    print("Skipping cleaning for video ID {video_id}...")
    cleaned_transcripts.append({"video_id": video_id, "text": full_text})
  else:
    print(f"Cleaning transcript for video ID {video_id}...")
    resp = ollama.chat(
            model="llama3.1:8b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: {full_text}"},
            ]
        )

    if resp:
      cleaned_text = clean_text(resp['message']['content'])
      cleaned_transcripts.append({"video_id": video_id, "text": cleaned_text})
      print(f"Successfully cleaned transcript for {video_id}.")
    else:
      print(f"Ollama did not return a response for video ID {video_id}. Using original text as fallback.")
      cleaned_transcripts.append({"video_id": video_id, "text": full_text})

with open(PATH_CLEANED, 'w', encoding='utf-8') as f:
    json.dump(cleaned_transcripts, f, ensure_ascii=False, indent=2)